In [392]:
%reset -f

# ---------------------------------------------------------------------------
# SETUP OG IMPORTS
# ---------------------------------------------------------------------------
from pathlib import Path
import sys
from sympy import *

candidates = [Path.cwd(), *Path.cwd().parents]
SUPPORT_ROOT = next(
    (
        p / "support_material"
        for p in candidates
        if (p / "support_material" / "scripts" / "control").exists()
    ),
    None,
)
if SUPPORT_ROOT is None:
    SUPPORT_ROOT = next((p for p in candidates if (p / "scripts" / "control").exists()), None)
if SUPPORT_ROOT is None:
    raise RuntimeError("Kan ikke finde support_material/scripts/control.")
if str(SUPPORT_ROOT) not in sys.path:
    sys.path.insert(0, str(SUPPORT_ROOT))

from scripts.control import (
    analyze_transfer_function,
    bode_to_transfer,
    closed_loop_analysis_from_coefficients,
    closed_loop_characteristic,
    closed_loop_poles,
    design_lag,
    design_pi_lead,
    design_pi_lead_at_crossover,
    evaluate_transfer_function,
    feedforward_analysis,
    find_stable_gain_ranges,
    frequency_response_point,
    ideal_disturbance_feedforward,
    nyquist_point_analysis,
    phase_margin_from_point,
    second_order_analysis,
    second_order_characteristics,
    solve_lag_beta,
    solve_stability_interval_by_boundary,
    steady_state_error_analysis,
    transfer_function_poles,
    unity_feedback_step_error,
)

print(f"Klar. Supportmateriale: {SUPPORT_ROOT}")

s, K = symbols("s K", real=True)
X1, X2, F = symbols("X1 X2 F")

Klar. Supportmateriale: c:\Users\sehes\GitHub\2026-spring-formelsamlinger\Linear control design 1\support_material


In [393]:
# ---------------------------------------------------------------------------
# TRANSFERFUNKTIONER, POLER, FEEDBACK OG RESPONS
# ---------------------------------------------------------------------------
# evaluate_transfer_function(numerator, denominator, omega)
# Input: koefficientlister i faldende potenser af s og frekvens omega [rad/s].
# Brug naar: beregn G(j*omega) fra en kendt transferfunktion.
# Eksempel: 1/(s+1) ved omega=1
# evaluate_transfer_function([1], [1, 1], 1)
#
# frequency_response_point(numerator, denominator, omega)
# Input: samme som evaluate_transfer_function.
# Returnerer: kompleks vaerdi, real/imag, magnitude, dB, fase og warnings.
# Brug naar: du vil have alle Bode-/Nyquist-tal ved en valgt frekvens.
# Eksempel: frequency_response_point([1], [1, 1], 1)
#
# transfer_function_poles(denominator)
# Input: naevnerkoefficienter i faldende potenser af s.
# Brug naar: find poler efter at transferfunktionen er udledt.
# Eksempel: 1/(s+1)^2
# transfer_function_poles([1, 2, 1])
#
# analyze_transfer_function(G, variable=s)
# Input: symbolsk transferfunktion G(s), eventuelt med parameteren K.
# Returnerer: poler, nulpunkter, DC gain, orden, type og stabilitetsdata.
# Brug naar: en udledt transferfunktion skal analyseres symbolsk samlet.
# Eksempel: K = symbols("K", positive=True); analyze_transfer_function(K/(s**2 + 5*s + K), s)
#
# closed_loop_poles(numerator, denominator, proportional_gain=1.0, feedback_gain=1.0)
# Input: G(s)=num/den samt gain for et negativt feedback-loop.
# Brug naar: kontroller stabilitet for et foreslaaet P-gain.
# Eksempel: F21 Q11, G(s)=120/(s^3+43*s^2+120*s), Kp=25
# closed_loop_poles([120], [1, 43, 120, 0], proportional_gain=25)
#
# closed_loop_analysis_from_coefficients(numerator, denominator, proportional_gain=1.0, feedback_gain=1.0)
# Input: G(s)=num/den og gain for negativ feedback.
# Returnerer: karakteristisk polynomium, poler, stabilitet og closed-loop DC-gain.
# Brug naar: du vil checke mere end bare polerne for et foreslaaet P-gain.
# Eksempel: closed_loop_analysis_from_coefficients([120], [1, 43, 120, 0], 25)
#
# closed_loop_characteristic(forward, feedback=1, variable=s, negative_feedback=True)
# Input: symbolsk forward- og feedback-transferfunktion samt feedbackfortegn.
# Brug naar: udled karakteristisk polynomium foer stabilitetsanalyse.
# Eksempel: K = symbols("K", real=True); closed_loop_characteristic(K/(s*(s+5)), variable=s)
#
# unity_feedback_step_error(numerator, denominator, proportional_gain=1.0)
# Input: G(s)=num/den og Kp for E/R=1/(1+Kp*G).
# Brug naar: stationaer fejl for unit step i verificeret negativ unity feedback.
# Eksempel: F21 Q16
# unity_feedback_step_error([1224], [1, 30, 257, 612], proportional_gain=2)
#
# steady_state_error_analysis(numerator, denominator, proportional_gain=1.0, input_type="step")
# Input: standard negativ unity feedback med L(s)=Kp*G(s), input_type step/ramp/parabolic.
# Returnerer: systemtype, fejlkonstanter Kp/Kv/Ka, closed-loop poler og stationaer fejl.
# Brug naar: opgaven handler om stationaer referencefejl i standardloop.
# Eksempel: steady_state_error_analysis([1], [1, 0], 2, input_type="ramp")
#
# second_order_characteristics(denominator)
# Input: andenordensnaevner [a2, a1, a0] for a2*s^2+a1*s+a0.
# Returnerer: omega_n, zeta og procent overshoot.
# Brug naar: standard andenordens steprespons.
# Eksempel: F21 Q9 ved K=20
# second_order_characteristics([1, 5, 20])
#
# second_order_analysis(denominator)
# Input: samme andenordensnaevner.
# Returnerer: omega_n, zeta, overshoot, poler, omega_d, peak time og settling estimates.
# Brug naar: du vil have flere tidsrespons-tal end den gamle korte helper giver.
# Eksempel: second_order_analysis([1, 5, 20])

# ---------------------------------------------------------------------------
# BODE, NYQUIST OG CONTROLLERDESIGN
# ---------------------------------------------------------------------------
# bode_to_transfer(dc_gain_db, poles=None, zeros=None, variable=s)
# Input: DC-gain i dB samt positive pol- og nulpunkt-knaekfrekvenser [rad/s].
# Brug naar: opstil G(s) fra et afleaest Bode-asymptotediagram.
# Eksempel: bode_to_transfer(20, poles=[10, 150], zeros=[100], variable=s)
#
# phase_margin_from_point(real_part, imaginary_part)
# Input: Nyquist-punkt paa enhedscirklen ved gain crossover.
# Brug naar: beregn phase margin fra et numerisk afleaest punkt.
# Eksempel: F21 Q14
# phase_margin_from_point(0.134, -0.99)
#
# nyquist_point_analysis(real_part, imaginary_part)
# Input: manuelt afleaest Nyquist-punkt.
# Returnerer: magnitude, dB, fase, phase margin, afstand til -1 og unit-circle warning.
# Brug naar: punktet skal checkes, ikke kun PM beregnes.
# Eksempel: nyquist_point_analysis(0.134, -0.99)
#
# design_pi_lead(numerator, denominator, omega_c=None, phase_margin_deg=None,
#                n_i=None, tau_i=None, alpha=None, tau_d=None, Kp=None)
# Regulator: C(s)=Kp*((tau_i*s+1)/(tau_i*s))*((tau_d*s+1)/(alpha*tau_d*s+1)).
# PI-led: (tau_i*s+1)/(tau_i*s). Lead-led: (tau_d*s+1)/(alpha*tau_d*s+1), normalt 0<alpha<1.
# Sammenhaenge: n_i=omega_c*tau_i og, naar leadmaksimum placeres ved omega_c,
# tau_d=1/(omega_c*sqrt(alpha)).
# Brug naar: en eksamensopgave giver nogle af omega_c, phase margin, n_i, tau_i,
# alpha, tau_d eller Kp, og du vil designe resten eller checke en foreslaaet regulator.
# Haandterer bl.a.: klassisk design fra omega_c/PM/n_i, direkte tau_i, givet alpha,
# givet tau_d+alpha, givet Kp, samt ren check-mode naar regulatorparametrene er kendte.
# Returnerer: Kp, tau_i, n_i, alpha, tau_d, target-checks ved omega_c, faktisk crossover,
# phase margin, gain margin og warnings. Kp beregnes saa |L(j*omega_c)|=1, hvis Kp ikke er givet.
# Wrapper: design_pi_lead_at_crossover(num, den, omega_c, phase_margin_deg, n_i)
# kalder design_pi_lead(...) og bevarer den gamle API.
# Eksempler:
# den = np.polymul(np.polymul([5, 1], [1, 0.2, 0.6]), [0.01, 1])
# design_pi_lead_at_crossover([0.7, 0.35], den, 10, 45, 8)        # gammel F21 Q18-form
# design_pi_lead([0.7, 0.35], den, omega_c=10, phase_margin_deg=45, n_i=8)
# design_pi_lead([0.7, 0.35], den, omega_c=10, phase_margin_deg=45, tau_i=0.8)
# design_pi_lead([0.7, 0.35], den, omega_c=10, alpha=0.08, n_i=8)
# design_pi_lead([0.7, 0.35], den, omega_c=10, Kp=200, tau_i=0.8, alpha=0.08, tau_d=0.354)
#
# solve_lag_beta(lag_phase_deg, n_i)
# Input: kraevet negativ Lag-fase i grader og Ni=omega_c*tau_i.
# Brug naar: beta skal findes i et P-Lead-Lag design.
# Eksempel: F21 Q17
# solve_lag_beta(-8.9193, 3)
#
# design_lag(lag_phase_deg=None, n_i=None, omega_c=None, tau_i=None, beta=None)
# Input: lagfase og n_i, eller givet beta; omega_c kan bruges til tau_i/n_i.
# Returnerer: beta, lagfase, tau_i, n_i, lag-zero/pole-frekvenser og warnings.
# Brug naar: P-Lead-Lag-opgaven skal designes eller checkes bredere end beta alene.
# Eksempel: design_lag(-8.9193, n_i=3, omega_c=10)
#
# ideal_disturbance_feedforward(plant_numerator, plant_denominator,
#                               disturbance_numerator, disturbance_denominator,
#                               disturbance_sign)
# Input: G(s), D(s) og disturbancefortegn (+1 eller -1) fra blokdiagrammet.
# Returnerer: Fd(s)-koefficienter samt proper/stable-status.
# Brug naar: nominal dynamisk feed-forward for en maalt disturbance.
# Eksempel: F21 Q20, disturbancebidraget er -D(s)*d(s)
# ideal_disturbance_feedforward([10.5, 21], [1, 4, 21], [1], [0.01, 1], -1)
#
# feedforward_analysis(plant_numerator, plant_denominator,
#                      disturbance_numerator, disturbance_denominator,
#                      disturbance_sign)
# Input: samme som ideal_disturbance_feedforward.
# Returnerer: ideal Fd(s), poler, nulpunkter, relative degree, proper/stable og warnings.
# Brug naar: du skal vurdere om den ideelle feed-forward faktisk kan realiseres.
# Eksempel: feedforward_analysis([10.5, 21], [1, 4, 21], [1], [0.01, 1], -1)

# ---------------------------------------------------------------------------
# STABILITETSINTERVAL FOR ET PARAMETERAFHAENGIGT KARAKTERISTISK POLYNOMIUM
# ---------------------------------------------------------------------------
# find_stable_gain_ranges(characteristic, gain, s)
# Input: p(s,K)=0 eller en ligning som 1+K*G(s)=0, samt reelle K og s.
# Funktionen samler automatisk en broek og bruger taelleren som p(s,K).
# Metode: loeser p(j*w,K)=0 og tester polerne mellem de fundne graenser.
# Output: marginale (K,w)-punkter og aabne intervaller med alle poler i LHP.
# Graensepunkterne er IKKE asymptotisk stabile, fordi de har poler paa jw-aksen.
# Brug kun naar p(s,K) er udledt, og graden i s ikke aendrer sig med K.
#
# s, K = symbols("s K", real=True)
# p = (s + 1)**3 + K                 # Direkte polynomium
# p = 1 + K / (s + 1)**3            # Samme opgave som 1+K*G(s)=0
# stability = find_stable_gain_ranges(p, K, s)
# print("Marginale punkter (K, w):", stability["boundary_points"])
# print("Stabile K-intervaller:", stability["stable_gain_intervals"])
# print("Stabile intervaller for K > 0:", stability["positive_stable_gain_intervals"])
# For dette eksempel: -1 < K < 8, og hvis K skal vaere positiv: 0 < K < 8.
# Skift blot linjen med p = ... ud med karakteristisk polynomium fra din opgave.
# solve_stability_interval_by_boundary(char_poly, gain_symbol, variable=s)
# Input: samme karakteristiske polynomium, kompakt wrapper omkring boundary-metoden.
# Brug naar: du kun vil have boundary equations, marginale gains og stabile intervaller.
# Eksempel: solve_stability_interval_by_boundary((s+1)**3 + K, K, s)
#
# solve_stability_interval_by_boundary(char_poly, gain_symbol, variable=s)
# Input: symbolsk karakteristisk polynomium og en reel gain-parameter.
# Brug naar: samme j*w-metode oenskes med et kompakt outputdictionary.
# Eksempel: solve_stability_interval_by_boundary((s + 1)**3 + K, K, s)

In [394]:
# opg 1

K, K_m, K_b = symbols("K K_m K_b")

DC = K_m/(s + 0.01)

X = DC/(1 + DC*K_b)

G = ((1/s)*X*K)/(1 + ((1/s)*X*K))

G.simplify().expand()

K*K_m/(K*K_m + K_b*K_m*s + s**2 + 0.01*s)

In [395]:
# opg 2

second_order_analysis([1, 2, 1])

{'omega_n': 1.0,
 'zeta': 1.0,
 'overshoot_percent': 0.0,
 'overshoot_fraction': 0.0,
 'poles': array([-1., -1.]),
 'omega_d': 0.0,
 'peak_time': None,
 'settling_time_2_percent': 4.0,
 'settling_time_5_percent': 3.0,
 'damping_class': 'critically damped'}

In [396]:
# opg 4
omegan = 5
PO = 10

zeta = -ln(PO/100)/sqrt(pi**2 + ln(PO/100)**2)

tp = pi/(omegan*sqrt(1 - zeta**2))

(pi/tp).evalf()

4.03278974784895

In [397]:
# opg 7
A = 1/(s - 2)
B = (s - 2)/(s - 7)
C = 2/(s + 2)
D = -1/(s + 2)

G = A*B/(1 + B*(C + D))

analyze_transfer_function(G)

{'G(s)': (s + 2)/(s**2 - 4*s - 16),
 'numerator': s + 2,
 'denominator': s**2 - 4*s - 16,
 'zeros_exact': {-2: 1},
 'poles_exact': {2 - 2*sqrt(5): 1, 2 + 2*sqrt(5): 1},
 'zeros_numeric': [(-2+0j)],
 'poles_numeric': [(6.47213595499958+0j), (-2.472135954999579+0j)],
 'dc_gain': -1/8,
 'order': 2,
 'numerator_order': 1,
 'system_type': 0,
 'stable': False,
 'stability_text': 'ustabil',
 'y(0+)': 0,
 'y(inf)': -1/8,
 'settling_time_2_percent': None,
 'settling_time_1_percent': None,
 'settling_time_text': 'ikke beregnet',
 'normalized_G': (s + 2)/(s**2 - 4*s - 16),
 'normalized_numerator': s + 2,
 'normalized_denominator': s**2 - 4*s - 16,
 'static_gain': -1/8,
 'relative_degree': 1,
 'proper': True,
 'strictly_proper': True,
 'biproper': False,
 'analysis_warnings': [],
 'numerical_tolerance': 1e-07,
 'is_symbolic_exact': False,
 'used_numeric_fallbacks': ['numpy.roots',
  'numeric_step_response',
  'numeric_impulse_response'],
 'gain_crossover_frequency': None,
 'phase_at_gain_crossover

In [398]:
# opg 9

PO = 1.23
ts = 1

zeta = -ln(PO/100)/sqrt(pi**2 + ln(PO/100)**2)
zeta = zeta.evalf()

omegan = 4 / (zeta * ts)

zeta, omegan

(0.813728864633526, 4.91564226593026)

In [399]:
# opg 10

G1 = 4/(s + 10)
G2 = 1/(s + 2)
G3 = (s + 2)/(s + 10)

G = (G1 + G2*G3)/(1 - (G1 + G2*G3))

print(G.simplify())
analyze_transfer_function(G)

5/(s + 5)


{'G(s)': 5/(s + 5),
 'numerator': 5,
 'denominator': s + 5,
 'zeros_exact': {},
 'poles_exact': {-5: 1},
 'zeros_numeric': [],
 'poles_numeric': [(-5+0j)],
 'dc_gain': 1,
 'order': 1,
 'numerator_order': 0,
 'system_type': 0,
 'stable': True,
 'stability_text': 'asymptotisk stabil',
 'y(0+)': 0,
 'y(inf)': 1,
 'settling_time_2_percent': 0.7824049803364689,
 'settling_time_1_percent': 0.9210342608875457,
 'settling_time_text': 'unit-step settling time relativt til slutvaerdien',
 'normalized_G': 5/(s + 5),
 'normalized_numerator': 5,
 'normalized_denominator': s + 5,
 'static_gain': 1,
 'relative_degree': 1,
 'proper': True,
 'strictly_proper': True,
 'biproper': False,
 'analysis_warnings': [],
 'numerical_tolerance': 1e-07,
 'is_symbolic_exact': False,
 'used_numeric_fallbacks': ['numpy.roots',
  'numeric_step_response',
  'numeric_impulse_response'],
 'gain_crossover_frequency': None,
 'phase_at_gain_crossover_deg': None,
 'phase_margin_deg': None,
 'phase_margin_text': 'ingen gain c

In [400]:
# opg 11

10**(16.4/20)

6.606934480075959

In [401]:
# opg 15

-0.6356 * 1.57

-0.9978920000000001

In [402]:
# opg 16

omegac = 45

C1 = (0.44 * s + 1)/(0.001*s + 1)
C2 = (450 * s + 1)/(2.5*s + 1)
C3 = (0.0022 * s + 1)/(0.22*s + 1)
C4 = (0.22 * s + 1)/(0.0022*s + 1)
C5 = (142.3 * s + 1)/(14.23*s + 1)

display(
    C1.subs(s, (I*omegac)).evalf(),
    C2.subs(s, (I*omegac)).evalf(),
    C3.subs(s, (I*omegac)).evalf(),
    C4.subs(s, (I*omegac)).evalf(),
    C5.subs(s, (I*omegac)).evalf()
)

1.88717846361119 + 19.7150769691375*I

179.985857907523 + 1.59098540362243*I

0.01999899000101 - 0.098990001009999*I

1.96088140138503 + 9.70587274126288*I

9.99997805141017 + 0.0140547794977905*I

In [403]:
# opg 17
Ni = 3
alpha = 0.02
beta = 10
gammaM = 70
omegac = 25.4921

taui = Ni / omegac
taud = 1 / (omegac * sqrt(alpha))

KP = 12

C = KP * ((taud*s+1)/(alpha*taud*s+1)) * ((taui * s + 1)/(taui * s + (1 / beta)))

G = bode_to_transfer(dc_gain_db=-15, poles=[5, 5, 100])

print(G.simplify())
#Vi ønsker ved 110 grader er gain 0
#Vi ved at G ved -167.4143 er gain -40.9645
#KP = 10**(gainchange/20)

250*10**(1/4)/((s + 5)**2*(s + 100))


In [404]:
# opg 17

Ni = 3
alpha = 0.02
beta = 10
gammaM = 70/360 * 2*pi
omegac = symbols('omega_c', Real=True)


eq = Eq(-pi + gammaM , arg(G.subs(s, (I*omegac))) + atan(Ni*(1 - beta)/(1 + beta*Ni**2)) + asin((1 - alpha)/(1 + alpha)))

#solve(eq)

In [408]:
# opg 18

alpha = 0.01

omegac = (-135*pi/180).evalf()

taud = 1/(omegac*sqrt(alpha))

G = 3.652/(s*(s + 1)*s + 5)

C = (taud*s + 1)/(alpha*taud*s + 1)

G = 3.652 / ((I*omegac)*((I*omegac) + 1)*(I*omegac) + 5)
C = (taud*(I*omegac) + 1)/(alpha*taud*(I*omegac) + 1)
(1/G*C).evalf()
sqrt(35.4079569580185**2 + 5.61205752996774**2)

35.8499456856012

In [406]:
val = -180 + 45

eq = Eq(val, arg(G.subs(s, (I*omegac)))/pi*180)

solve(eq)

[]

In [407]:
eq = Eq(abs(G.subs(s, (I*omegac))) , 1)

solve(eq)

[]

In [413]:
# opg 17
Ni = 3
alpha = 0.02
beta = 10
gammaM = 70
omegac = symbols('omega_c', Real=True)

taui = Ni / omegac
taud = 1 / (omegac * sqrt(alpha))

KP = 12

C = KP * ((taud*s+1)/(alpha*taud*s+1)) * ((taui * s + 1)/(taui * s + (1 / beta)))

G = bode_to_transfer(dc_gain_db=-15, poles=[5, 5, 100])

(arg(G.subs(s, (I*omegac))) + 180).evalf()

print(G)

250*10**(1/4)/((s + 5)**2*(s + 100))


In [414]:
250*10**(1/4)/(((I*omegac) + 5)**2*((I*omegac) + 100))

444.569852509731/((I*omega_c + 5)**2*(I*omega_c + 100))